### Data Ingestion

In [1]:
### Document Structure

from langchain_core.documents import Document

d:\Computer Science\Switch\Gen AI\RAG\.venv-1\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
doc = Document(
    page_content="This is the content of the RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"John Doe",
        "date_created":"2026-01-01"
    }
)

doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'John Doe', 'date_created': '2026-01-01'}, page_content='This is the content of the RAG')

In [3]:
### Create a simple text file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
### We will create two text files with python script
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction""",
    "../data/text_files/data_science.txt":"""Data Science with Python""",
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files create!")

Sample text files create!


In [5]:
### Now, we will load the files using   "TextLoader"   from langchain_core.document_loaders
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")

document = loader.load()
print(document)
## In the below output we can see the output in the form of LangChain Document. It has metadata as well as page_content

d:\Computer Science\Switch\Gen AI\RAG\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction')]


In [6]:
### Now, we will load the files using   "DirectoryLoader"
from langchain_community.document_loaders import DirectoryLoader
dir_loader = DirectoryLoader(
    "../data/text_files", 
    glob="**/*.txt", ## pattern to match files 
    loader_cls= TextLoader, ## which loader class to use for loading the files. Because we are loading text files from the directory
    loader_kwargs={"encoding":"utf-8"},
    show_progress=False
)

documents = dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\data_science.txt'}, page_content='Data Science with Python'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction')]

In [7]:
### Now, we will load the PDF files 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
dir_loader = DirectoryLoader(
    "../data/pdf_files", 
    glob="**/*.pdf", ## pattern to match files 
    loader_cls= PyMuPDFLoader, ## which loader class to use for loading the files. Because we are loading pdf files from the directory
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-13T16:10:25+00:00', 'source': '..\\data\\pdf_files\\Resume_BTech.pdf', 'file_path': '..\\data\\pdf_files\\Resume_BTech.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-13T16:10:25+00:00', 'trapped': '', 'modDate': 'D:20230813161025Z', 'creationDate': 'D:20230813161025Z', 'page': 0}, page_content='Mohammad Ausaf Shah\n+91-9682138656 | ms2396@srmist.edu.in | Linkdin | Github\nEducation\nSRM Institute of Science and Technology\nKattankulathur, Tamil Nadu\nB.Tech in Computer Science and engineering - 9.11 CGPA\nJuly. 2020 – Currently\nBurn Hall School\nSrinagar, J&K\nClass XII - 92.2%\n2019 – 2020\nBurn Hall School\nSrinagar, J&K\nClass X - 92%\n2017 – 2018\nExperience\nWeb Development Internship\nFebruary 2022 – March 2022\nBooksApp\nRemote\n∗Developed a landing page for a Cargo company based in Dubai using H

In [8]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### Embedding and VectorStore DB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer # the embedding model will be available insie this
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity # cosine similarity will be used while retrieving from the vector database

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): # in this init method we are initializing the embedding manager and loading the model. The model name that we are giving is "all-MiniLM-L6-v2" which is a popular model for generating sentence embeddings, it is available fromHuggingFace. It converts text into vectors and we get around 384 dimensions
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for SentenceTransformer 
        """
        self.model_name = model_name # Initialising the model name
        self.model = None
        self._load_model() 
    
    def _load_model(self): # the _ means protected function, it will be called only in this class
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model :{self.model_name}")
            self.model = SentenceTransformer(self.model_name) # loading the model using sentence transformer
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:  # this method takes text, which is basically List of strings and finally returns a numpy array.
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            Numpy array of embeddings with shape (len(texts), embedding_dimension).
        """
        if not self.model:
            raise ValueError("Model not loaded.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True) # generating the embeddings using the model
            print(f"Generated embeddings generated successfully with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

    def get_embedding_dimension(self) -> int:
        """Get embedding dimension of the model."""
        if not self.model:
            raise ValueError("Model not loaded.")
        return self.model.get_sentence_embedding_dimension()
    

### initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model :all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3247.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension:384


### Vector Store

In [11]:
import os
import chromadb
class VectoreStore:
    """Manages document embeddings using ChromaDB vector store."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"): ## We are giving the collection name and persistant directory for the vector store. Persistant_directory means whatever vectoreStore is created, we will store it in the hard disk.
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory to persist the ChromaDB collection.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store() ## Means this function will initialise the vector store
    
    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True) # We are specifying the durectory. If it alreadys exists, fine. Otherwise it will create it
            self.client = chromadb.PersistentClient(path=self.persist_directory) # We are creating a client which will have reference to the Chromadb VectorStore

            # Get or create collection
            self.collection = self.client.get_or_create_collection( # Creating the collection. Basically, where we will store the vectors inside the VectorStore
                name=self.collection_name,
                metadata={"description": "PDF Document embedding for RAG"}
            )
            print(f"Vector store initialized. Collection:{self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing VectorStore: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray): # The embedding is coming from generate_embeddings method of the embedding manager
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain documents.
            embeddings: Corresponding embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
    
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.matadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            document_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist()) 
        
        # Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
    
vectorstore = VectoreStore()
vectorstore
        

Vector store initialized. Collection:pdf_documents
Existing documents in collection: 0
